In [0]:
WITH geo_normalizada AS (
    SELECT
        geolocation_zip_code_prefix AS zip_prefix,
        UPPER(TRIM(geolocation_state)) AS uf,
        TRANSLATE(
            LOWER(TRIM(geolocation_city)),
            'áàâãäéèêëíìîïóòôõöúùûüç',
            'aaaaaeeeeiiiiooooouuuuc'
        ) AS cidade,
        geolocation_lat AS lat,
        geolocation_lng AS lng
    FROM workspace.olist_landing.bronze_geolocation
)
SELECT
    zip_prefix,
    uf,
    cidade,
    PERCENTILE_APPROX(lat, 0.5) AS latitude,
    PERCENTILE_APPROX(lng, 0.5) AS longitude
FROM geo_normalizada
GROUP BY zip_prefix, uf, cidade

In [0]:
select 1001
1001 from workspace.olist_landing.bronze_geolocation



In [0]:
select * from workspace.olist_landing.bronze_geolocation where geolocation_zip_code_prefix = 6311

In [0]:
select distinct seller_state, seller_city , seller_zip_code_prefix
from workspace.olist_silver.seller
where seller_state = 'SP'
and seller_city like 'carapicuiba%'



In [0]:
create or replace temporary view modelo_entrega as
select * from (
With distancia as (
select  
o.order_id, 
datediff(Cast(o.order_delivered_customer_date as date), Cast(o.order_delivered_carrier_date as date)) as delivery_days,
Cast(o.order_approved_at as date) as order_approved ,
Cast(o.order_delivered_carrier_date as date) as carrier_delivery_dt,
Cast(o.order_delivered_customer_date as date) as customer_delivery_dt,
Cast(o.order_estimated_delivery_date as date) as estimated_delivery_dt,
geo_cus.lat as customer_lat,
geo_cus.lng as cutomer_lng,
geo_cus.city as customer_city,
geo_cus.state as customer_state,
geo_sel.state as seller_state,
geo_sel.city as seller_city,
geo_sel.lat as seller_lat,
geo_sel.lng as seller_lng,
item.seller_id as seller_id,
CASE
    WHEN geo_sel.lat IS NOT NULL AND geo_sel.lng IS NOT NULL
        AND geo_cus.lat IS NOT NULL AND geo_cus.lng IS NOT NULL
    THEN 2 * 6371 * ASIN(SQRT(LEAST(1.0,
                POWER(SIN(RADIANS(geo_cus.lat - geo_sel.lat) / 2), 2)
                + COS(RADIANS(geo_sel.lat)) * COS(RADIANS(geo_cus.lat))
                * POWER(SIN(RADIANS(geo_cus.lng - seller_lng) / 2), 2)
            )))
END AS distancia_km,
CASE
    WHEN customer_delivery_dt > estimated_delivery_dt THEN 1
    ELSE 0
END AS atrasou
from workspace.olist_silver.orders_customer o
inner join workspace.olist_silver.customers c
    on o.customer_id = c.customer_id
    and c.customer_unique_id = o.customer_unique_id
inner join  (
    SELECT DISTINCT order_id, seller_id
    FROM workspace.olist_silver.items
    WHERE seller_id IS NOT NULL
) AS item
    ON item.order_id = o.order_id
left join workspace.olist_silver.seller s
    on item.seller_id = s.seller_id  
left join workspace.olist_silver.geolocation geo_cus
    on c.customer_zip_code_prefix  = geo_cus.zip_code_prefix 
left join workspace.olist_silver.geolocation geo_sel
    on seller_zip_code_prefix  = geo_sel.zip_code_prefix      
where o.order_status = 'delivered'
  AND o.order_approved_at IS NOT NULL
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL)

SELECT
    order_id,
    customer_city,
    customer_state,
    estimated_delivery_dt,
    customer_delivery_dt,
    carrier_delivery_dt,
    DATEDIFF(estimated_delivery_dt,order_approved) as prazo_prometido_dias,
    delivery_days,
    COUNT(DISTINCT seller_id) AS qtd_vendedores,
    COUNT(DISTINCT CASE WHEN distancia_km IS NOT NULL THEN seller_id END)
        AS qtd_vendedores_com_distancia,
    ROUND(MAX(distancia_km), 1) AS distancia_max_km,
    ROUND(AVG(distancia_km), 1) AS distancia_media_km,
    atrasou
FROM distancia
GROUP BY order_id, customer_city, customer_state, estimated_delivery_dt, customer_delivery_dt, carrier_delivery_dt,
    delivery_days, DATEDIFF(estimated_delivery_dt,order_approved), atrasou)

In [0]:
select * from modelo_entrega where order_id = '36e694cf4cbc2a4803200c35e84abdc4'

In [0]:
select order_id, count(atrasou)
from modelo_entrega
group by order_id
having count(atrasou)  > 1 


--17332
--13248
--155905
--117173



In [0]:
WITH pontos AS (
    SELECT DISTINCT
        geolocation_zip_code_prefix AS zip_prefix,
        UPPER(TRIM(geolocation_state)) AS uf,
        geolocation_lat AS lat,
        geolocation_lng AS lng
    FROM workspace.olist_landing.bronze_geolocation
    WHERE geolocation_zip_code_prefix IS NOT NULL
      AND geolocation_state IS NOT NULL
      AND geolocation_lat BETWEEN -90 AND 90
      AND geolocation_lng BETWEEN -180 AND 180
)
SELECT
    zip_prefix,
    uf,
    PERCENTILE_APPROX(lat, 0.5) AS latitude,
    PERCENTILE_APPROX(lng, 0.5) AS longitude,
    COUNT(*) AS pontos_distintos
FROM pontos
GROUP BY zip_prefix, uf

In [0]:
select * from workspace.olist_landing.bronze_sellers

In [0]:
select i.seller_id , p.product_category_name, product_description_lenght , s.*
from workspace.olist_silver.items i
inner join workspace.olist_silver.product p
on p.product_id = i.product_id
inner join workspace.olist_silver.seller s
on i.seller_id = s.seller_id
where i.order_id  = 'bddbca2d48a961d74fe81de8febf9a73'

In [0]:

Select distinct i.order_id, count(distinct i.seller_id) 
from workspace.olist_silver.items i
where 
group by  i.order_id
having count(distinct i.seller_id)  > 1

In [0]:
select * from workspace.olist_silver.customers where customer_unique_id in ('4c93744516667ad3b8f1fb645a3116a4')

In [0]:
select distinct customer_id, customer_unique_id , Count(order_id)
from  workspace.olist_silver.orders_customer o
where customer_unique_id = '4c93744516667ad3b8f1fb645a3116a4'
group by  customer_id, customer_unique_id

In [0]:
select count(*), year(o.order_approved_at), o.order_status from  workspace.olist_landing.bronze_orders o
left join workspace.olist_landing.bronze_items i
on o.order_id =  i.order_id
where i.order_id is null
group  by year(o.order_approved_at), o.order_status


In [0]:
select * from  workspace.olist_silver.items




In [0]:
describe extended workspace.olist_landing.bronze_items

In [0]:
select * from read_files(
  '/Volumes/workspace/olist_landing/raw_files/olist_customers_dataset.csv'
)

In [0]:
select * from read_files('/Volumes/workspace/olist_landing/raw_files/olist_sellers_dataset.csv')

In [0]:
%sql 
with
amostra as (SELECT
    review_comment_message AS comentario,
    sentimento AS sentimento_ia,
    tema AS tema_ia,
    problema AS problema_ia
FROM workspace.olist_gold.fact_review
WHERE LOWER(TRIM(review_source)) = 'original'
  AND review_score = 2
  AND tema IS NOT NULL
ORDER BY review_id
LIMIT 100) 

SELECT comentario, 
ai_translate(comentario,'en') as traducao , 
ai_translate(sentimento_ia,'en') as sentimento_ia_query , 
ai_analyze_sentiment(comentario) AS ai_sentiment,
ai_classify(comentario,array('entrega','produto','arvore', 'atraso','brinquedo' )) as ai_classify
,tema_ia
FROM amostra

In [0]:
SELECT * FROM system.billing.usage LIMIT 100